In [1]:
# Deep Learning framework
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary

# Audio processing
import torchaudio
import torchaudio.transforms as T
import librosa

# Pre-trained image models
# import timm

# Play the audio in Jupyter notebook
from IPython.display import Audio
import pandas as pd
import os
import numpy as np

from scripts.dataset import AudioDataset
from models.crnn_parallel import CRNNNetwork as NNetwork

if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(DEVICE)

cuda


In [2]:
def train_single_epoch(model, data_loader, criterion, optimizer):
    for batch_idx, (input, labels) in enumerate(data_loader):
        input, labels = input.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        # calculate loss
        outputs = model.forward(input)
        loss = criterion(outputs, labels)

        # backpropagate error and update weights
        loss.backward()
        optimizer.step()
    
    return loss



def train(model, data_loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        loss = train_single_epoch(model, data_loader, criterion, optimizer)
        if epoch % (epochs // 10) == 0:
            print(f"iteration {epoch}: loss {loss.item()} | accuracy: {accuracy(train_set)}")

In [3]:
AUDIO_DIR = "audios/labeled/"

dict_genres = {'positive': 0, 'negative': 1}

reverse_map = {v: k for k, v in dict_genres.items()}

data = []

for label in dict_genres.keys():
    for folder in os.listdir(AUDIO_DIR + label):
        for file in os.listdir(AUDIO_DIR + label + "/" + folder):
            file_path = AUDIO_DIR + label + "/" + folder + "/" + file
            positive = int(label == "positive")
            negative = int(label == "negative")
            data.append((file_path, positive,
                        negative))

file_path, positive, negative = zip(*data)
df = pd.DataFrame({"file_path": file_path, "positive": positive, "negative": negative})

In [4]:
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 0.001
SAVE = False

dataset = AudioDataset(df)

train_dataloader = create_data_loader(dataset, BATCH_SIZE)

# construct model and assign it to device
model = NNetwork().to(DEVICE)
summary(model.cuda(), (1, 64, 44))

# initialise loss funtion + optimiser
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                                lr=LEARNING_RATE)

# train model
train(model, train_dataloader, criterion, optimizer, EPOCHS)

# save model
if SAVE:
    torch.save(model.state_dict(), "feedforwardnet.pth")
    print("Trained feed forward net saved at feedforwardnet.pth")

Epoch 1


AssertionError: GRU: Expected input to be 2-D or 3-D but received 4-D tensor